# Episode > 0 Smoke Test

This notebook generates SimulateTS data and runs FC2 training for episode_id > 0 with randomized model weights.


In [9]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if (ROOT / 'config').exists():
    sys.path.append(str(ROOT))
elif (ROOT / 'Project1' / 'DL-AP' / 'config').exists():
    sys.path.append(str(ROOT / 'Project1' / 'DL-AP'))
else:
    for p in ROOT.parents:
        if (p / 'config').exists():
            sys.path.append(str(p))
            break

import torch
import numpy as np

from config import Config
from models import PolicyValueModel, SDFFC1Combined, FC2Model
from training.episode import Episode

# randomize seeds (non-deterministic)
torch.seed()
np.random.seed(None)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def _reinit(m):
    if hasattr(m, 'reset_parameters'):
        m.reset_parameters()


In [10]:
# init models with random weights
models = {
    'policy_value': PolicyValueModel().to(device),
    'sdf_fc1': SDFFC1Combined().to(device),
    'fc2': FC2Model().to(device),
}
for m in models.values():
    if m is not None:
        m.apply(_reinit)

optimizers = {
    'policy_value': torch.optim.Adam(models['policy_value'].parameters(), lr=1e-3),
    'sdf_fc1': torch.optim.Adam(models['sdf_fc1'].parameters(), lr=1e-3),
    'fc2': torch.optim.Adam(models['fc2'].parameters(), lr=1e-4),
}

episode = Episode(
    models=models,
    optimizers=optimizers,
    config=Config,
    device=device,
    episode_id=1,
)


In [11]:
summary = episode.run_episode(
    n_epochs=2,
    batch_size=1,
    log_interval=1,
    n_paths=1,
    group_size=50,
    n_branches=2,
    train_modules=['fc2'],
    simulate_kwargs={'horizon': 2},
)

print('episode_id:', summary.get('episode_id'))
print('fc2 summary:', summary['module_summaries'].get('fc2'))
print('df shape:', getattr(episode, 'df', None).shape if getattr(episode, 'df', None) is not None else None)


Simulating paths: 100%|██████████| 1/1 [00:00<00:00, 48.26it/s]


RuntimeError: The expanded size of the tensor (1000) must match the existing size (1150) at non-singleton dimension 0.  Target sizes: [1000, 5].  Tensor sizes: [1150, 5]

# FC2Loss Debug: raw batch -> converted -> pipe


In [12]:
import pandas as pd
import numpy as np
import torch
from config import Config
from training.episode import convert_tree_fast, trim_child_only_ids
from experiments.fill_fullN_entrants import fill_df_to_fullN
from losses.FC2losspipe import FC2LossPipe

device = _device if '_device' in globals() else torch.device('cpu')
full_N = getattr(Config, 'FULL_N', 1000)
entry_num = getattr(Config, 'ENTRY_NUM', None)


In [ ]:
# 1) raw batch (SimulateTS output)
df_raw = episode.df
print('raw shape:', df_raw.shape)
print('raw columns:', list(df_raw.columns))
print('raw branch counts:', df_raw['branch'].value_counts().sort_index().to_dict())
print('raw t unique:', sorted(df_raw['t'].unique())[:10])
print('raw path count:', df_raw['path'].nunique())
print('raw IDs per path (nunique) describe:')
print(df_raw.groupby('path')['ID'].nunique().describe())
print('raw rows per path describe:')
print(df_raw.groupby('path').size().describe())
dup_key = df_raw.duplicated(subset=['path','t','branch','ID'])
print('raw duplicate (path,t,branch,ID) rows:', int(dup_key.sum()))
print('has Entry:', 'Entry' in df_raw.columns, 'has entry:', 'entry' in df_raw.columns)
display(df_raw.head())


raw shape: (370, 27)
raw columns: ['path', 't', 'branch', 'ID', 'entry', 'b', 'z', 'ETA', 'i', 'x', 'Hatcf', 'LnKF', 'K', 'M', 'Q', 'P0', 'PI', 'Bar_i', 'Bar_z', 'P', 'bp0', 'bpI', 'bp', 'Y', 'I', 'Phi', 'C']
raw branch counts: {-1: 110, 0: 130, 1: 130}
raw t unique: [0, 1, 2]
raw path count: 1
raw IDs per path (nunique) describe:
count     1.0
mean     90.0
std       NaN
min      90.0
25%      90.0
50%      90.0
75%      90.0
max      90.0
Name: ID, dtype: float64
raw rows per path describe:
count      1.0
mean     370.0
std        NaN
min      370.0
25%      370.0
50%      370.0
75%      370.0
max      370.0
dtype: float64
raw duplicate (path,t,branch,ID) rows: 0
has Entry: False has entry: True


,path,t,branch,ID,entry,b,z,ETA,i,x,...,Bar_i,Bar_z,P,bp0,bpI,bp,Y,I,Phi,C
0,0,0,-1,24a6e859679e4119b691c5f89bab03d8,1.0,0.038894,0.220083,0.0,0.208031,-1.953963,...,0.543838,3.379951e-15,0.666418,0.506902,0.453217,0.477706,0.046582,0.035118,6.294000e-16,0.011464
1,0,0,-1,fd2939feb15b4590823bd62a777b4e61,1.0,0.245861,0.936330,0.0,0.258505,-1.953963,...,0.543083,3.370449e-15,0.666475,0.506868,0.453496,0.477882,0.069044,0.030637,5.259171e-16,0.038406
2,0,0,-1,12c67ef4539146a384820a1ca5145c17,1.0,0.027602,0.333265,0.0,0.114577,-1.953963,...,0.543768,3.377051e-15,0.666435,0.506933,0.452972,0.477591,0.013817,0.005750,1.695613e-16,0.008067
3,0,0,-1,f419b6a3696d433f82b849cf57aa57ca,1.0,0.356711,1.199036,0.0,0.088795,-1.953963,...,0.543310,3.362743e-15,0.666520,0.506801,0.453260,0.477711,0.013619,0.001977,8.593700e-17,0.011642
4,0,0,-1,17f9b7c3ec3c49049c64d3f64b6f48a8,1.0,0.320059,-1.148085,0.0,0.271521,-1.953963,...,0.541907,3.444117e-15,0.666042,0.506657,0.452851,0.477499,0.008479,0.031521,4.072419e-16,-0.023043


In [ ]:
# 2) converted batch (convert_tree_fast)
df_conv = convert_tree_fast(df_raw)
df_conv = trim_child_only_ids(df_conv)
df_conv = promote_child_only_ids(df_conv)
df_conv = trim_child_only_ids(df_conv)
print('conv shape:', df_conv.shape)
print('conv branch counts:', df_conv['branch'].value_counts().sort_index().to_dict())
print('conv t unique:', sorted(df_conv['t'].unique()))
print('conv path count:', df_conv['path'].nunique())
print('conv IDs per path (nunique) describe:')
print(df_conv.groupby('path')['ID'].nunique().describe())
print('conv rows per path describe:')
print(df_conv.groupby('path').size().describe())
display(df_conv.head())


conv shape: (370, 30)
conv branch counts: {0: 110, 1: 130, 2: 130}
conv t unique: ['t', 't+1_0', 't+1_1']
conv path count: 2
conv IDs per path (nunique) describe:
count     2.000000
mean     75.000000
std       7.071068
min      70.000000
25%      72.500000
50%      75.000000
75%      77.500000
max      80.000000
Name: ID, dtype: float64
conv rows per path describe:
count      2.000000
mean     185.000000
std       21.213203
min      170.000000
25%      177.500000
50%      185.000000
75%      192.500000
max      200.000000
dtype: float64


,path,t,branch,ID,entry,b,z,ETA,i,x,...,bp0,bpI,bp,Y,I,Phi,C,path_ori,t_ori,branch_ori
0,0,t,0,24a6e859679e4119b691c5f89bab03d8,1.0,0.038894,0.220083,0.0,0.208031,-1.953963,...,0.506902,0.453217,0.477706,0.046582,0.035118,6.294000e-16,0.011464,0,0,-1
1,0,t,0,fd2939feb15b4590823bd62a777b4e61,1.0,0.245861,0.936330,0.0,0.258505,-1.953963,...,0.506868,0.453496,0.477882,0.069044,0.030637,5.259171e-16,0.038406,0,0,-1
2,0,t,0,12c67ef4539146a384820a1ca5145c17,1.0,0.027602,0.333265,0.0,0.114577,-1.953963,...,0.506933,0.452972,0.477591,0.013817,0.005750,1.695613e-16,0.008067,0,0,-1
3,0,t,0,f419b6a3696d433f82b849cf57aa57ca,1.0,0.356711,1.199036,0.0,0.088795,-1.953963,...,0.506801,0.453260,0.477711,0.013619,0.001977,8.593700e-17,0.011642,0,0,-1
4,0,t,0,17f9b7c3ec3c49049c64d3f64b6f48a8,1.0,0.320059,-1.148085,0.0,0.271521,-1.953963,...,0.506657,0.452851,0.477499,0.008479,0.031521,4.072419e-16,-0.023043,0,0,-1


In [ ]:
# 3) fill_df_to_fullN
df_filled = fill_df_to_fullN(df_conv, full_N=full_N, device=device, entry_num=entry_num)
print('filled shape:', df_filled.shape)
print('filled rows per path describe:')
print(df_filled.groupby('path').size().describe())
print('filled IDs per path (nunique) describe:')
print(df_filled.groupby('path')['ID'].nunique().describe())
# child-only IDs (exist in child branches but not in parent)
parent_ids = df_filled[df_filled['branch']==0].groupby('path')['ID'].apply(set)
child_ids = df_filled[df_filled['branch']>0].groupby('path')['ID'].apply(set)
child_only = (child_ids - parent_ids).apply(lambda s: len(s))
print('child-only IDs per path describe:')
print(child_only.describe())
display(df_filled.head())


filled shape: (6040, 31)
filled rows per path describe:
count       2.0
mean     3020.0
std         0.0
min      3020.0
25%      3020.0
50%      3020.0
75%      3020.0
max      3020.0
dtype: float64
filled IDs per path (nunique) describe:
count       2.0
mean     1020.0
std         0.0
min      1020.0
25%      1020.0
50%      1020.0
75%      1020.0
max      1020.0
Name: ID, dtype: float64
child-only IDs per path describe:
count     2.0
mean     20.0
std       0.0
min      20.0
25%      20.0
50%      20.0
75%      20.0
max      20.0
Name: ID, dtype: float64


,path,t,branch,ID,entry,b,z,ETA,i,x,...,bpI,bp,Y,I,Phi,C,path_ori,t_ori,branch_ori,Entry
0,0,t,0,24a6e859679e4119b691c5f89bab03d8,1.0,0.038894,0.220083,0.0,0.208031,-1.953963,...,0.453217,0.477706,0.046582,0.035118,6.294000e-16,0.011464,0.0,0.0,-1.0,NaN
1,0,t,0,fd2939feb15b4590823bd62a777b4e61,1.0,0.245861,0.936330,0.0,0.258505,-1.953963,...,0.453496,0.477882,0.069044,0.030637,5.259171e-16,0.038406,0.0,0.0,-1.0,NaN
2,0,t,0,12c67ef4539146a384820a1ca5145c17,1.0,0.027602,0.333265,0.0,0.114577,-1.953963,...,0.452972,0.477591,0.013817,0.005750,1.695613e-16,0.008067,0.0,0.0,-1.0,NaN
3,0,t,0,f419b6a3696d433f82b849cf57aa57ca,1.0,0.356711,1.199036,0.0,0.088795,-1.953963,...,0.453260,0.477711,0.013619,0.001977,8.593700e-17,0.011642,0.0,0.0,-1.0,NaN
4,0,t,0,17f9b7c3ec3c49049c64d3f64b6f48a8,1.0,0.320059,-1.148085,0.0,0.271521,-1.953963,...,0.452851,0.477499,0.008479,0.031521,4.072419e-16,-0.023043,0.0,0.0,-1.0,NaN


In [ ]:
print(df_filled.groupby(['path','t','branch']).size())

path  t      branch
0     t      0         1000
      t+1_0  1         1010
      t+1_1  2         1010
1     t      0         1000
      t+1_0  1         1010
      t+1_1  2         1010
dtype: int64


In [ ]:
# 4) emulate FC2LossPipe merge to find oversize paths
parent = df_filled[df_filled.branch == 0].reset_index(drop=True)
child1 = df_filled[df_filled.branch == 1].reset_index(drop=True)
child2 = df_filled[df_filled.branch == 2].reset_index(drop=True)
df1 = pd.merge(parent, child1, on=['ID'], suffixes=('', '_child1'), how='outer')
df2 = pd.merge(df1, child2, on=['ID'], suffixes=('', '_child2'), how='outer')
df2 = df2.sort_values(['path','b'], ascending=[True, True], na_position='last', kind='mergesort')
group_sizes = df2.groupby('path').size()
print('df2 rows per path describe:')
print(group_sizes.describe())
print('max df2 rows per path:', int(group_sizes.max()))
bad_paths = group_sizes[group_sizes > full_N]                     .sort_values(ascending=False)
print('paths exceeding full_N:', bad_paths.to_dict())


In [ ]:
# 5) try FC2LossPipe and catch failure point
try:
    pipe = FC2LossPipe(df=df_conv, full_N=full_N, entry_num=entry_num, device=device)
    print('pipe.P_s_full:', pipe.P_s_full.shape)
    print('pipe.Children_s_full:', pipe.Children_s_full.shape)
except Exception as e:
    print('FC2LossPipe error:', repr(e))
